In [4]:
!pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

In [5]:
#hide
from fastbook import *
from fastai.vision.all import *

Download images of cats and dogs and train the model

In [ ]:
path = untar_data(URLs.PETS)/'images'

def is_cat(x): return x[0].isupper()

dls = ImageDataLoaders.from_name_func(path, get_image_files(path), is_cat, valid_pct=0.2, seed=42, item_tfms=Resize(224))

learn = cnn_learner(dls, resnet34, metrics=error_rate)
learn.fine_tune(1)

Test the model

In [ ]:
uploader = widgets.FileUpload()
uploader

In [ ]:
img = PILImage.create(uploader.data[0])
is_cat, _, probs = learn.predict(img)

print(f"Is this a cat: {is_cat}")
print(f"Probability its a cat: {probs[1]:.6f}")

print(f"Predictions for: {learn.dls.vocab} are: {probs[0]:.6f} {probs[1]:.6f}")

Segmentation of objects in a scene

In [30]:
path = untar_data(URLs.CAMVID_TINY)

dls = SegmentationDataLoaders.from_label_func(
    path, 
    get_image_files(path/'images'),
    label_func = lambda o: path/'labels'/f'{o.stem}_P{o.suffix}',
    bs=8,
    codes= np.loadtxt(path/'codes.txt', dtype=str))

learn = unet_learner(dls, resnet34)
learn.fine_tune(8)

Downloading: "https://download.pytorch.org/models/resnet34-333f7ec4.pth" to /home/.cache/torch/hub/checkpoints/resnet34-333f7ec4.pth


  0%|          | 0.00/83.3M [00:00<?, ?B/s]

Show resulst

In [ ]:
learn.show_results(max_n=6, figsize=(20, 50))

Movie Sentiment review

In [31]:
from fastai.text.all import *

dls = TextDataLoaders.from_folder(untar_data(URLs.IMDB), valid='test')

learn = text_classifier_learner(dls, AWD_LSTM, drop_mult=0.5, metrics=accuracy)
learn.fine_tune(4, 1e-2)

Predict the sentiment

In [32]:
print(learn.dls)
learn.predict("I really liked that movie")

('pos', tensor(1), tensor([1.1240e-04, 9.9989e-01]))

In [33]:
learn.predict("Meeeh")

('pos', tensor(1), tensor([0.0589, 0.9411]))

Predict if someone is a high income earner using his social-ecoonomic background

In [34]:
from fastai.tabular.all import *

path = untar_data(URLs.ADULT_SAMPLE)

dls = TabularDataLoaders.from_csv(
    path/'adult.csv',
    path=path,
    y_names="salary",
    cat_names = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race'],
    cont_names = ['age', 'fnlwgt', 'education-num'],
    procs = [Categorify, FillMissing, Normalize])

learn = tabular_learner(dls, metrics=accuracy)
learn.fit_one_cycle(3)

Show the results

In [36]:
learn.show_results()

,workclass,education,marital-status,occupation,relationship,race,education-num_na,age,fnlwgt,education-num,salary,salary_pred
0,5.0,12.0,6.0,15.0,2.0,5.0,1.0,-0.553003,1.367546,-0.422345,0.0,0.0
1,5.0,6.0,1.0,8.0,2.0,5.0,1.0,0.474638,0.133502,-2.380773,0.0,0.0
2,5.0,10.0,3.0,2.0,1.0,5.0,1.0,1.208668,-0.280965,1.144397,1.0,1.0
3,8.0,9.0,3.0,5.0,1.0,5.0,1.0,0.181026,1.640250,0.361026,0.0,1.0
4,5.0,12.0,3.0,8.0,1.0,5.0,1.0,1.282071,-1.481922,-0.422345,0.0,0.0
5,5.0,12.0,7.0,7.0,2.0,5.0,1.0,1.575683,-0.106738,-0.422345,0.0,0.0
6,5.0,10.0,5.0,2.0,4.0,5.0,2.0,-0.773212,1.026768,-0.030659,0.0,0.0
7,5.0,7.0,3.0,4.0,1.0,5.0,1.0,1.575683,-0.686074,-1.989088,0.0,0.0
8,5.0,10.0,5.0,9.0,2.0,5.0,1.0,-0.039183,-0.567925,1.144397,0.0,0.0


Movie recommndedation system

In [6]:
from fastai.collab import *

path = untar_data(URLs.ML_SAMPLE)

dls = CollabDataLoaders.from_csv(path/'ratings.csv')

learn = collab_learner(dls, y_range=(0.5, 5.5))
learn.fine_tune(10)

epoch,train_loss,valid_loss,time
0,1.376203,1.359422,00:00
1,1.252138,1.180762,00:00
2,1.017709,0.880817,00:00
3,0.798113,0.741172,00:00
4,0.688681,0.708689,00:00
5,0.648084,0.697439,00:00
6,0.631074,0.693731,00:00
7,0.608035,0.691561,00:00
8,0.609987,0.691219,00:00
9,0.607285,0.691045,00:00


Show results

In [7]:
learn.show_results()

,userId,movieId,rating,rating_pred
0,63.0,48.0,5.0,2.935433
1,70.0,36.0,4.0,4.040259
2,65.0,92.0,4.5,4.266208
3,47.0,98.0,5.0,4.345913
4,4.0,83.0,3.5,4.335626
5,4.0,38.0,4.5,4.221953
6,59.0,60.0,5.0,4.432968
7,86.0,82.0,4.0,3.797124
8,55.0,86.0,4.5,3.959364
